In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import random

In [11]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: cpu | GPU: CPU


In [19]:
train_x = pd.read_csv('train_x.csv')
train_y = pd.read_csv('train_y.csv')
test_x  = pd.read_csv('test_x.csv')

id_col = train_x.columns[0]
test_ids = test_x[id_col].values

X = train_x.drop(columns=[id_col]).values.astype(np.float32)
X_test = test_x.drop(columns=[id_col]).values.astype(np.float32)
y_col = train_y.columns[-1]
y = train_y[y_col].values.astype(np.float32)

MIN_YEAR, MAX_YEAR = int(y.min()), int(y.max())
N_CLASSES = MAX_YEAR - MIN_YEAR + 1
print(f'Years: {MIN_YEAR}..{MAX_YEAR} ({N_CLASSES} classes)')

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=SEED)

Years: 1922..2011 (90 classes)


In [20]:
scaler = StandardScaler()
X_tr  = scaler.fit_transform(X_tr).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

In [21]:
y_mean, y_std = y_tr.mean(), y_tr.std()
y_tr_n  = ((y_tr  - y_mean) / y_std).astype(np.float32)
y_val_n = ((y_val - y_mean) / y_std).astype(np.float32)

In [22]:
BATCH = 1024

def make_loader(Xa, ya, shuffle):
    ds = TensorDataset(torch.from_numpy(Xa), torch.from_numpy(ya))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

train_loader_reg = make_loader(X_tr, y_tr_n, True)
val_loader_reg   = make_loader(X_val, y_val_n, False)


y_tr_cls  = (y_tr  - MIN_YEAR).astype('int64')
y_val_cls = (y_val - MIN_YEAR).astype('int64')
train_loader_cls = make_loader(X_tr, y_tr_cls, True)
val_loader_cls   = make_loader(X_val, y_val_cls, False)

test_tensor = torch.from_numpy(X_test).to(device)
years_tensor = torch.arange(MIN_YEAR, MAX_YEAR + 1, dtype=torch.float32, device=device)


In [23]:
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class YearRegressor(nn.Module):
    def __init__(self, in_dim, hidden=512):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.SiLU(),
        )
        self.res1 = ResBlock(hidden, 0.3)
        self.res2 = ResBlock(hidden, 0.2)
        self.head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.res1(x)
        x = self.res2(x)
        return self.head(x).squeeze(-1)

In [24]:
class YearClassifier(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=512):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.SiLU(),
        )
        self.res1 = ResBlock(hidden, 0.3)
        self.res2 = ResBlock(hidden, 0.2)
        self.head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, n_classes)
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.res1(x)
        x = self.res2(x)
        return self.head(x)

In [25]:
def train_model(model, train_loader, val_loader, criterion,
                epochs=60, lr=1e-3, wd=1e-5, patience=10,
                label_smoothing=0.0, is_classification=False):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    if label_smoothing > 0 and is_classification:
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    best_val = float('inf')
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            tr_loss += loss.item() * xb.size(0)
        tr_loss /= len(train_loader.dataset)

        model.eval()
        val_mse = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                out = model(xb)
                if is_classification:
                    probs = torch.softmax(out, dim=1)
                    preds = (probs * years_tensor).sum(dim=1)
                    true_years = yb.to(device).float() + MIN_YEAR
                else:
                    preds = out * y_std + y_mean
                    true_years = yb.to(device) * y_std + y_mean
                val_mse += ((preds - true_years) ** 2).sum().item()
        val_mse /= len(val_loader.dataset)

        scheduler.step()

        if val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d} | train loss {tr_loss:8.4f} | val MSE {val_mse:8.2f} | best {best_val:8.2f}')

        if wait >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model, best_val


In [26]:
reg_model = YearRegressor(X.shape[1]).to(device)
reg_model, reg_val = train_model(
    reg_model, train_loader_reg, val_loader_reg,
    criterion=nn.HuberLoss(delta=1.0),
    epochs=60, lr=1e-3, wd=1e-5, patience=10,
    is_classification=False
)
print(f'Regression best val MSE: {reg_val:.2f}')


=== REGRESSION ===
  Epoch   1 | train loss   0.3434 | val MSE   107.09 | best   107.09
  Epoch   5 | train loss   0.2394 | val MSE    89.80 | best    89.11
  Epoch  10 | train loss   0.1984 | val MSE    95.93 | best    89.11
  Early stopping at epoch 14
Regression best val MSE: 89.11


In [27]:
cls_model = YearClassifier(X.shape[1], N_CLASSES).to(device)
cls_model, cls_val = train_model(
    cls_model, train_loader_cls, val_loader_cls,
    criterion=nn.CrossEntropyLoss(),
    epochs=60, lr=1e-3, wd=1e-5, patience=10,
    label_smoothing=0.05, is_classification=True
)
print(f'Classification best val MSE: {cls_val:.2f}')


=== CLASSIFICATION ===
  Epoch   1 | train loss   4.2592 | val MSE   804.92 | best   804.92
  Epoch   5 | train loss   3.0660 | val MSE    89.65 | best    89.65
  Epoch  10 | train loss   2.5215 | val MSE    90.15 | best    86.46
  Epoch  15 | train loss   2.0247 | val MSE    92.35 | best    86.46
  Early stopping at epoch 18
Classification best val MSE: 86.46


In [28]:
if cls_val < reg_val:
    cls_model.eval()
    with torch.no_grad():
        logits = cls_model(test_tensor)
        probs = torch.softmax(logits, dim=1)
        preds = (probs * years_tensor).sum(dim=1).cpu().numpy()
else:
    reg_model.eval()
    with torch.no_grad():
        preds_n = reg_model(test_tensor).cpu().numpy()
        preds = preds_n * y_std + y_mean

preds_int = np.clip(np.rint(preds).astype(int), MIN_YEAR, MAX_YEAR)

submission = pd.DataFrame({'index': test_ids, 'year': preds_int})
submission.to_csv('submission.csv', index=False)
print('\nSaved submission.csv')
print(submission.head())
print('Prediction stats:', preds_int.min(), preds_int.max(), preds_int.mean())


>>> Using CLASSIFICATION model

Saved submission.csv
   index  year
0   3416  1998
1  18991  2002
2  11105  1995
3  18902  1999
4  18958  2000
Prediction stats: 1972 2008 1996.7706666666666
